# 多智能体

上一章的工作流里，谁做什么、什么时候做，都是代码事先安排好的。这一章，我们把决定权交给智能体：让它在需要的时候，**自己决定**去找哪个智能体帮忙。

为什么要用好几个智能体，而不是一个什么都会的智能体呢？

- **专注**：每个智能体只有一份简短的系统消息、几个用得上的工具，更容易把自己那件事做好。一个智能体如果同时有几十个工具、一大段要求，反而容易糊涂。
- **上下文隔离**：每个智能体有自己独立的对话历史。帮手在完成任务时查了多少资料、试了多少次，都留在它自己那里，只把最终结果交回来。主智能体的对话历史因此保持简洁。
- **好维护**：哪个智能体出了问题，就单独改它，不影响其他的。

当然，多智能体也有代价：调用大模型的次数更多，花费更多，出了问题也更难排查。所以还是那条原则：**单个智能体能做好的，就不要拆**。

这一章介绍三种多智能体的合作方式：

1. **子智能体**：主智能体把任务交给帮手，帮手完成后把结果交回来；
2. **智能体对话**：两个智能体你一句、我一句地交流；
3. **交接**：一个智能体把和用户的对话**转交**给另一个智能体。

In [ ]:
import asyncio
import inspect
import json
import os
import re
import threading
import time
import uuid
from abc import ABC, abstractmethod
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints
from zoneinfo import ZoneInfo

import anthropic
import numpy as np
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from mcp import Client
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None, thinking: List[dict] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []
        self.thinking = thinking or []  # the model's reasoning blocks, passed back to the API unchanged

    def thinking_text(self) -> str:
        return "\n".join(block.get("thinking", "") for block in self.thinking).strip()

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls], "thinking": self.thinking}

In [ ]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

In [ ]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])], data.get("thinking", []))
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [ ]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)

In [ ]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )

In [ ]:
@tool
def get_current_time(timezone: Annotated[str, "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


print(json.dumps(get_current_time.parameters, indent=2))
print(get_current_time("America/Toronto"))

In [ ]:
@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


print(json.dumps(add_days.parameters, indent=2))
print(add_days("2026-11-10", 100))

In [ ]:
class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it

In [ ]:
class MCPConnection:
    """A synchronous wrapper around the async mcp.Client (Part 6 will make the whole Agent async).

    The connection lives in one long-running coroutine on a background event loop;
    every call is submitted to that loop and we wait for its result.
    """

    def __init__(self, url: str):  # the URL of a remote MCP server
        self.loop = asyncio.new_event_loop()
        threading.Thread(target=self.loop.run_forever, daemon=True).start()
        ready = threading.Event()
        self.session = asyncio.run_coroutine_threadsafe(self._session(url, ready), self.loop)
        ready.wait()
        if self.session.done():  # connecting failed: raise the error here
            self.session.result()

    async def _session(self, url: str, ready: threading.Event):
        self.closed = asyncio.Event()
        try:
            async with Client(url) as client:
                self.client = client
                ready.set()
                await self.closed.wait()  # keep the connection open until close()
        finally:
            ready.set()

    def _run(self, coroutine):
        return asyncio.run_coroutine_threadsafe(coroutine, self.loop).result()

    def list_tools(self):
        return self._run(self.client.list_tools()).tools

    def call_tool(self, name: str, arguments: dict):
        return self._run(self.client.call_tool(name, arguments))

    def close(self):
        self.loop.call_soon_threadsafe(self.closed.set)
        self.session.result()
        self.loop.call_soon_threadsafe(self.loop.stop)

In [ ]:
class MCPTool(Tool):
    """A tool that lives on an MCP server: same interface as Tool, but run() goes over MCP."""

    def __init__(self, connection: MCPConnection, mcp_tool):
        super().__init__(mcp_tool.name, mcp_tool.description or "", mcp_tool.input_schema, function=None)
        self.connection = connection

    def run(self, arguments: dict) -> str:
        result = self.connection.call_tool(self.name, arguments)
        text = "\n".join(block.text for block in result.content if block.type == "text")
        if result.is_error and not text.startswith("Error"):
            text = f"Error: {text}"  # same convention as ToolRegistry.run()
        return text


def load_mcp_tools(connection: MCPConnection) -> List[Tool]:
    return [MCPTool(connection, t) for t in connection.list_tools()]

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096,
             output_schema: dict = None):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self, effort: str = None):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"
        self.effort = effort  # "none", "low", "medium", "high" or "xhigh"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096,
             output_schema: dict = None):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        if self.effort:
            kwargs["reasoning_effort"] = self.effort  # the model thinks, but only the answer comes back
        if output_schema:  # the answer must be JSON that follows the schema
            kwargs["response_format"] = {"type": "json_schema",
                                         "json_schema": {"name": "output", "strict": True, "schema": output_schema}}
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self, effort: str = None):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-5-5"  # a reasoning model: thinking is on by default
        self.effort = effort  # "low", "medium", "high" (default), "xhigh" or "max"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=16000,
             output_schema: dict = None):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        kwargs["thinking"] = {"type": "adaptive", "display": "summarized"}  # return a summary of the thinking
        if self.effort:
            kwargs["output_config"] = {"effort": self.effort}
        if output_schema:  # the answer must be JSON that follows the schema
            kwargs.setdefault("output_config", {})["format"] = {"type": "json_schema", "schema": output_schema}
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = list(msg.thinking)  # thinking blocks first, unchanged: required during tool use
                content += [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        thinking = [
            {"type": "thinking", "thinking": block.thinking, "signature": block.signature} if block.type == "thinking"
            else {"type": "redacted_thinking", "data": block.data}
            for block in response.content if block.type in ("thinking", "redacted_thinking")
        ]
        return AssistantMessage(text, tool_calls, thinking)

In [ ]:
class EmbeddingProvider(ABC):
    model: str = ""  # name of the embedding model, saved together with an index

    @abstractmethod
    def embed(self, texts: List[str]) -> np.ndarray:
        """Return one vector per text, shape (len(texts), dimension)."""
        pass

In [ ]:
class OpenAIEmbedding(EmbeddingProvider):

    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.model = model

    def embed(self, texts: List[str]) -> np.ndarray:
        response = self.client.embeddings.create(model=self.model, input=texts)
        data = sorted(response.data, key=lambda d: d.index)  # one embedding per input, in order
        return np.array([d.embedding for d in data])

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between every row of a and every row of b, shape (len(a), len(b))."""
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

In [ ]:
class Chunk:

    def __init__(self, text: str, source: str):
        self.text = text
        self.source = source  # which document it comes from, used for citations in chapter 15

    def to_dict(self) -> dict:
        return {"text": self.text, "source": self.source}


def load_documents(folder: str) -> List[tuple]:
    """Return (source, text) for every Markdown file in the folder."""
    return [(path.name, path.read_text(encoding="utf-8")) for path in sorted(Path(folder).glob("*.md"))]

In [ ]:
def chunk_fixed(text: str, size: int = 200, overlap: int = 50) -> List[str]:
    """Cut the text every `size` characters; neighbouring chunks share `overlap` characters."""
    step = size - overlap
    return [text[start:start + size] for start in range(0, max(len(text) - overlap, 1), step)]

In [ ]:
def chunk_by_heading(text: str, source: str, max_size: int = 500) -> List[Chunk]:
    """One chunk per "## " section, prefixed with its headings; long sections are cut again."""
    chunks, title, heading, lines = [], "", "", []

    def flush():
        body = "\n".join(lines).strip()
        if body:
            prefix = " > ".join(h for h in (title, heading) if h)
            pieces = chunk_fixed(body, max_size, overlap=50) if len(body) > max_size else [body]
            chunks.extend(Chunk(f"{prefix}\n{piece}", source) for piece in pieces)
        lines.clear()

    for line in text.splitlines():
        if line.startswith("# "):
            flush()
            title, heading = line[2:].strip(), ""
        elif line.startswith("## "):
            flush()
            heading = line[3:].strip()
        else:
            lines.append(line)
    flush()
    return chunks

In [ ]:
class VectorIndex:

    def __init__(self, embedder: EmbeddingProvider):
        self.embedder = embedder
        self.chunks: List[Chunk] = []
        self.vectors = None  # shape (len(chunks), dimension)

    def add(self, chunks: List[Chunk], batch_size: int = 100):
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
        vectors = np.vstack([self.embedder.embed([c.text for c in batch]) for batch in batches])
        self.vectors = vectors if self.vectors is None else np.vstack([self.vectors, vectors])
        self.chunks += chunks

    def search(self, query: str, top_k: int = 3, min_score: float = 0.0) -> List[tuple]:
        if not self.chunks:
            return []
        scores = cosine_similarity(self.embedder.embed([query]), self.vectors)[0]
        best = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in best if scores[i] >= min_score]

    def save(self, folder: str):
        folder = Path(folder)
        folder.mkdir(exist_ok=True)
        np.save(folder / "vectors.npy", self.vectors)
        data = {"model": self.embedder.model, "chunks": [c.to_dict() for c in self.chunks]}
        (folder / "chunks.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, folder: str, embedder: EmbeddingProvider) -> "VectorIndex":
        folder = Path(folder)
        data = json.loads((folder / "chunks.json").read_text(encoding="utf-8"))
        if data["model"] != embedder.model:  # vectors of different models can not be compared
            raise ValueError(f"index was built with '{data['model']}', not '{embedder.model}'")
        index = cls(embedder)
        index.chunks = [Chunk(**c) for c in data["chunks"]]
        index.vectors = np.load(folder / "vectors.npy")
        return index

In [ ]:
class LongTermMemory:
    EXTRACT_PROMPT = """From the conversation below, extract facts about the user that are worth remembering
in future conversations, such as their name, background, preferences and goals.
Write one short fact per line, in the same language as the user.
Do not repeat facts that are already known. If there is nothing new worth remembering, answer NONE.

Already known:
{known}

User: {user}
Assistant: {assistant}"""

    def __init__(self, provider: LLMProvider, embedder: EmbeddingProvider, folder: str = "long_term_memory"):
        self.provider = provider
        self.folder = folder
        if Path(folder, "chunks.json").exists():
            self.index = VectorIndex.load(folder, embedder)  # memories from earlier sessions
        else:
            self.index = VectorIndex(embedder)

    def recall(self, query: str, top_k: int = 3, min_score: float = 0.25) -> List[str]:
        return [chunk.text for chunk, _ in self.index.search(query, top_k, min_score)]

    def remember(self, user: UserMessage, assistant: AssistantMessage) -> List[str]:
        known = self.recall(user.request, top_k=5)
        prompt = LongTermMemory.EXTRACT_PROMPT.format(
            known="\n".join(known) or "(nothing)", user=user.request, assistant=assistant.content)
        response = self.provider.call(SystemMessage("You extract long-term memories about the user."), [UserMessage(prompt)])

        facts = [line.strip("-* ").strip() for line in response.content.splitlines()]
        facts = [fact for fact in facts if fact and fact.upper() != "NONE"]
        if facts:
            today = datetime.now().strftime("%Y-%m-%d")
            self.index.add([Chunk(fact, source=today) for fact in facts])  # source: when it was remembered
            self.index.save(self.folder)
        return facts

    def facts(self) -> List[str]:
        return [chunk.text for chunk in self.index.chunks]

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, registry: ToolRegistry = None, max_steps: int = 5,
                 long_term: LongTermMemory = None):
        self.provider = provider
        self.system = system
        self.registry = registry or ToolRegistry()
        self.max_steps = max_steps
        self.long_term = long_term
        self.store = MemoryStore()
        self.memory = Memory(provider)

    @staticmethod
    def show_thinking(response: AssistantMessage):
        if response.thinking_text():
            note("thinking", response.thinking_text())

    def run(self, message: str, system: SystemMessage = None) -> AssistantMessage:
        system = system or self.system
        memories = self.long_term.recall(message) if self.long_term else []
        if memories:  # put what we remember about the user into the system prompt
            for memory in memories:
                note("recall", memory)
            system = SystemMessage(system.summary() + "\n\nWhat you remember about the user:\n"
                                   + "\n".join(f"- {memory}" for memory in memories))
        tools = self.registry.list()
        turn = [UserMessage(message)]

        response = self.provider.call(system, self.memory.messages + turn, tools)
        self.show_thinking(response)
        turn.append(response)

        step = 0
        while response.tool_calls and step < self.max_steps:
            step += 1
            for call in response.tool_calls:
                result = self.registry.run(call)
                note(f"tool[{step}]", f"{call} -> {result}")
                turn.append(ToolResultMessage(call.id, call.name, result))
            response = self.provider.call(system, self.memory.messages + turn, tools)
            self.show_thinking(response)
            turn.append(response)

        if response.tool_calls:
            response.tool_calls = []
            response.content = response.content or f"(stopped after {self.max_steps} steps)"

        self.memory.add(*turn)
        self.store.save(self.memory)

        if self.long_term:  # extract new long-term facts from this turn
            for fact in self.long_term.remember(turn[0], response):
                note("remember", fact)
        return response

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            response = self.run(message, system)
            print(f"AI: {response.content}")

## 准备：一个查书的工具

先给学习助手准备一些关于这本书的笔记，放进第 14 章的 `VectorIndex`，再用第 11 章的 `@tool` 做成一个查书的工具 `search_book`。这和第 15 章的 `search_knowledge` 是一样的思路，只是资料直接写在代码里。

In [ ]:
provider = OpenAIProvider()

# a few notes about this book: the knowledge of our learning assistant
BOOK_NOTES = {
    "第1-4章": "第 1–4 章从调用大模型 API 开始，抽象出 LLMProvider 统一 OpenAI 和 Claude 的接口，"
              "用 Message 类表示对话消息，并写出最简单的 Agent 对话循环。",
    "第5-8章": "第 5–8 章讲对话记忆：保存历史消息，历史过长时压缩成摘要；把记忆保存为 JSON 文件，"
              "用 /new 开始新记忆、用 #id 加载旧记忆；最后实现流式输出。",
    "第9-12章": "第 9–12 章讲工具调用：模型请求工具，程序执行后把结果发回；工具循环一直进行到模型给出答案，"
               "max_steps 防止无限循环；@tool 装饰器和 ToolRegistry 管理工具；通过 MCP 连接远程工具服务器。",
    "第13-16章": "第 13–16 章讲 RAG：用嵌入向量表示文本，用余弦相似度做语义检索；文档分块、建立向量索引；"
                "检索增强生成并在答案中给出引用；用向量保存长期记忆。",
    "第17-20章": "第 17–20 章讲推理与规划：思维链提示和 ReAct；推理模型的 thinking；结构化输出与先计划后执行；"
                "生成、批评、修改的反思循环。",
    "第21-24章": "第 21–24 章讲多智能体与实际应用：工作流模式；子智能体、智能体对话和交接；人工确认、提示注入防御、"
                "输入输出检查；重试、日志和成本统计。",
    "MCP": "MCP（模型上下文协议）是一个开放协议，用统一的方式把工具提供给 AI 应用。客户端先通过 initialize 握手，"
           "再用 tools/list 发现工具、用 tools/call 调用工具；远程服务器使用 Streamable HTTP 传输。",
    "RAG": "RAG（检索增强生成）先从知识库检索相关内容，再把检索结果放进提示词，让模型据此回答；"
           "可以用 min_score 过滤掉不相关的结果。",
}

book_index = VectorIndex(OpenAIEmbedding())
book_index.add([Chunk(text, source) for source, text in BOOK_NOTES.items()])


@tool
def search_book(query: Annotated[str, "What to look up in the book"]) -> str:
    """Search the notes of the book 《从大模型到智能体》 (From LLM to Agent)."""
    results = book_index.search(query, top_k=2)
    return "\n\n".join(f"[{chunk.source}] {chunk.text}" for chunk, _ in results) or "Nothing found."

## 方式一：子智能体

### 把智能体包装成工具

怎样让一个智能体去找另一个智能体帮忙？最简单的办法：**把帮手包装成一个工具**。

回想第 11 章说过的：工具箱不关心工具是从哪里来的。第 12 章放进去了远程的 MCP 工具，第 15 章放进去了检索工具。现在，我们把一个完整的 `Agent` 也放进去。

`agent_as_tool` 把一个 `Agent` 包装成 `Tool`：

- 工具只有一个参数 `task`，是交给帮手的任务描述；
- 工具执行时，先给帮手一个**全新的记忆**，然后让它运行 `run(task)`，最后只把**最终回答**作为工具结果返回。

注意参数说明里写着：“子智能体看不到别的内容”。帮手只知道 `task` 里写的东西，看不到主智能体和小明的对话。所以主智能体必须把任务**写完整**，比如“为 MCP 出一道单选题”，而不是“为它出一道题”。

In [ ]:
def agent_as_tool(agent: Agent, name: str, description: str) -> Tool:
    """Wrap an Agent as a Tool: the main agent calls it like any other tool."""

    def run_subagent(task: str) -> str:
        note(name, f"start: {task}")
        agent.memory = Memory(agent.provider)  # a fresh, separate context for every task
        result = agent.run(task).content
        note(name, "done")
        return result  # only the final result goes back to the main agent

    return Tool(
        name=name,
        description=description,
        parameters={
            "type": "object",
            "properties": {"task": {"type": "string", "description": "A complete task description: the sub-agent sees nothing else"}},
            "required": ["task"],
        },
        function=run_subagent,
    )

### 三个帮手

我们创建三个各有专长的帮手：

| 帮手 | 擅长 | 工具 |
|---|---|---|
| `tutor`（辅导老师） | 讲解书中的概念 | `search_book` |
| `scheduler`（日程安排） | 推算学习日期 | `get_current_time`、`add_days` |
| `quizmaster`（出题人） | 出练习题 | 没有工具 |

然后用 `agent_as_tool` 把它们包装成工具，放进一个工具箱 `helpers`。工具的描述很重要：主智能体就是根据描述，来决定把什么任务交给谁。

In [ ]:
TUTOR = ("You are the tutor for the book 《从大模型到智能体》. Look things up with search_book, "
         "then explain simply, for beginners, in a few Chinese sentences.")
SCHEDULER = "You make study schedules with real dates. Use your tools for today's date and for counting days. Answer briefly in Chinese."
QUIZMASTER = "You write one multiple-choice question in Chinese about the given topic: 4 options, then the correct answer."

tutor = Agent(provider, SystemMessage(TUTOR), registry=ToolRegistry([search_book]))
scheduler = Agent(provider, SystemMessage(SCHEDULER), registry=ToolRegistry([get_current_time, add_days]))
quizmaster = Agent(provider, SystemMessage(QUIZMASTER))  # no tools

helpers = ToolRegistry([
    agent_as_tool(tutor, "tutor", "Explains concepts of the book to a beginner."),
    agent_as_tool(scheduler, "scheduler", "Works out study dates and weekdays."),
    agent_as_tool(quizmaster, "quizmaster", "Writes a multiple-choice practice question about a topic."),
])

### 协调者

最后创建主智能体，也就是**协调者**（coordinator）。它自己没有任何普通工具，工具箱里只有三个帮手。它的工作就是把任务分给合适的帮手，再把结果整合起来。

小明提出一个需要三个帮手配合的请求。观察协调者把哪些任务交给了谁，以及每个帮手内部又调用了哪些工具。

In [ ]:
coordinator = Agent(
    provider,
    SystemMessage("You are the coordinator of a learning assistant. Give subtasks to your helper agents and "
                  "combine their results. Each helper only sees the task you give it, so include everything it needs."),
    registry=helpers,
)

response = coordinator.run("我想在两周后复习完 MCP 这一章。先简单解释 MCP 是什么，再出一道练习题，最后告诉我两周后是几号、星期几。")
print("\nAI:", response.content)

### 上下文隔离

看一看协调者和 `tutor` 各自的对话历史。协调者只看到了帮手们的**最终回答**；`tutor` 查书的过程（`search_book` 的调用和返回的资料）都留在 `tutor` 自己的记忆里。

这就是**上下文隔离**。如果所有事情都由一个智能体来做，查资料、算日期的全部细节都会堆在同一个对话历史里，越来越长。分给子智能体以后，主智能体的对话历史始终保持简洁。

> Claude Code 也是这样做的：遇到需要大量搜索的任务，它会启动一个子智能体去搜索，子智能体读了很多文件，最后只把结论交回给主智能体。

In [ ]:
# context isolation: the coordinator only saw the helpers' final answers ...
print("coordinator:", len(coordinator.memory.messages), "messages")
print(coordinator.memory.history())

# ... while the details (the search_book call) stayed inside the tutor
print("\ntutor:", len(tutor.memory.messages), "messages")
print(tutor.memory.history())

## 方式二：智能体对话

第二种方式是让两个智能体**直接对话**：一个智能体的回答，就是另一个智能体收到的下一条消息。

我们创建一个“学生”智能体 `student`，让它扮演一个初学者：听完讲解以后，提一个追问；听懂了，就只回复“我明白了”。然后让它和 `tutor` 对话。

`conversation` 函数实现了这个过程：

- 每个智能体有自己的记忆，各自记着自己这一方的对话；
- 学生的问题交给老师，老师的回答交给学生，学生的追问再交给老师……
- 两个智能体如果一直聊下去，就停不下来了。所以必须有**停止条件**：学生说“我明白了”，或者聊满 `max_rounds` 轮。

In [ ]:
STUDENT = ("You are a beginner reading the book 《从大模型到智能体》. After each explanation, ask one short "
           "follow-up question in Chinese. When you understand, reply only: 我明白了")
student = Agent(provider, SystemMessage(STUDENT))


def conversation(student: Agent, tutor: Agent, question: str, max_rounds: int = 3) -> List[tuple]:
    """Two agents talk: each reply of one agent is the next message for the other."""
    for agent in (student, tutor):
        agent.memory = Memory(provider, max_messages=20)  # each agent keeps its own side of the talk
    transcript = []
    for round in range(max_rounds):
        print(f"[student] {question}\n")
        answer = tutor.run(question).content
        print(f"[tutor] {answer}\n")
        transcript.append((question, answer))
        question = student.run(answer).content
        if "我明白了" in question:  # a stop condition, or the two would talk forever
            print("[student] 我明白了")
            break
    return transcript


transcript = conversation(student, tutor, "什么是工具调用？")

对话结束以后，把整个过程整理成一份问答形式的 FAQ。这样，两个智能体的对话就变成了一份学习资料，小明可以拿来复习。

> 智能体对话还有很多用法，比如让两个智能体扮演正方和反方进行辩论，或者一个写代码、一个审查代码。第 20 章的反思，也可以看作出题者和老师之间的一种对话。

In [ ]:
text = "\n\n".join(f"Q: {q}\nA: {a}" for q, a in transcript)
faq = provider.call(SystemMessage("Turn this tutoring dialogue into a short FAQ in Chinese, as Q/A pairs."), [UserMessage(text)])
print(faq.content)

## 方式三：交接

子智能体的方式里，始终是协调者在和小明对话，帮手只在幕后工作。还有一种方式：**交接**（handoff），由一个智能体把和用户的对话**转交**给另一个智能体，之后由新的智能体直接和用户对话。就像打客服电话时，前台说“我帮您转接技术部门”，之后就是技术人员在和你说话了。

实现交接，同样用一个工具：`transfer_to`。智能体调用它，就表示“把对话交给某某”。工具做的事情很简单：把 `current["name"]`（当前负责和用户对话的智能体）改成新的名字。

我们创建三个智能体：

- `front_desk`（前台）：自己不回答问题，只负责转接；
- `tutor`：回答关于书的问题，遇到日期问题就转给 `scheduler`；
- `scheduler`：回答日期问题，遇到书的内容就转给 `tutor`。

`send` 函数把小明的消息交给当前负责的智能体。如果它在处理过程中调用了 `transfer_to`，就把**同一条消息**交给新的智能体再处理一次。为了防止几个智能体互相踢皮球，最多交接 `max_handoffs` 次。

> 我们的实现很简单：每个智能体有各自独立的记忆，接手的智能体看不到之前别人和小明的对话。真实的产品通常会把对话历史一起交接过去，让新的智能体了解前面聊过什么。

In [ ]:
agents = {}                       # name -> Agent, filled below
current = {"name": "front_desk"}  # who talks to the user right now


@tool
def transfer_to(agent_name: Annotated[str, "front_desk, tutor or scheduler"]) -> str:
    """Hand the conversation over to another agent, who then talks to the user directly."""
    current["name"] = agent_name
    return f"transferred to {agent_name}"


agents["front_desk"] = Agent(provider, SystemMessage(
    "You are the front desk of a learning assistant. Do not answer yourself: transfer the user "
    "to tutor (questions about the book) or scheduler (study plans and dates)."), registry=ToolRegistry([transfer_to]))
agents["tutor"] = Agent(provider, SystemMessage(TUTOR + " If the user asks about study plans or dates, transfer to scheduler."),
                        registry=ToolRegistry([search_book, transfer_to]))
agents["scheduler"] = Agent(provider, SystemMessage(SCHEDULER + " If the user asks about the content of the book, transfer to tutor."),
                            registry=ToolRegistry([get_current_time, add_days, transfer_to]))
for agent in agents.values():
    agent.memory = Memory(provider, max_messages=20)  # room for a whole conversation


def send(message: str, max_handoffs: int = 3) -> str:
    """One user message goes to the agent in charge; if it hands off, the new agent takes the message."""
    name = current["name"]
    response = agents[name].run(message)
    for _ in range(max_handoffs):
        if current["name"] == name:
            break
        name = current["name"]
        note("handoff", f"-> {name}")
        response = agents[name].run(message)
    return response.content

小明连续发三条消息，观察每条消息最终是由谁来回答的，以及什么时候发生了交接：

In [ ]:
for message in ["MCP 是什么？", "我想三天后复习这一章，那天是星期几？", "好的。再给我讲讲 RAG。"]:
    print("\nYOU:", message)
    print("AI:", send(message))
    print(f"   (now talking to: {current['name']})\n")

## 三种方式回顾

| 方式 | 谁和用户对话 | 结果怎么回来 | 适合 |
|---|---|---|---|
| 子智能体 | 始终是协调者 | 帮手把最终结果交回给协调者 | 任务可以拆分，需要汇总结果 |
| 智能体对话 | 两个智能体互相对话 | 对话记录本身就是结果 | 讨论、辩论、教学、审查 |
| 交接 | 由当前负责的智能体直接对话 | 不用交回，新的智能体接着聊 | 客服、按专业分工的助手 |

## 动手试试

- 给协调者再加一个帮手，比如用第 12 章的 DeepWiki 工具做一个 `researcher`，负责查开源项目的代码，然后问一个需要它帮忙的问题。
- 把学生的系统消息改成“你是一个很难被说服的学生”，看看对话会进行几轮。
- 在交接的例子里，让 `front_desk` 也能回答一些简单的问题，比如打招呼，不必每次都转接。

## 小结

- 多个智能体分工合作，可以让每个智能体更专注，并通过**上下文隔离**保持对话历史简洁。代价是调用次数更多。
- **子智能体**：把智能体包装成工具，主智能体像调用工具一样把任务交给帮手。交给帮手的任务要写完整。
- **智能体对话**：一个智能体的回答是另一个的输入，一定要有停止条件。
- **交接**：用一个工具把对话转交给另一个智能体，由它直接和用户对话。
- 工具箱是这一切的基础：本地函数、远程 MCP 工具、检索工具，现在连智能体本身，都可以作为工具放进去。

学习助手现在已经是一个智能体团队了。能力越大，责任也越大：如果它能发邮件、改文件，万一被人利用，或者自己理解错了，后果可能很严重。下一章，我们来讨论智能体的**安全**。